In [18]:
import stable_worldmodel as swm
from stable_worldmodel.policy import PlanConfig, WorldModelPolicy
from stable_worldmodel.solver import CEMSolver
from stable_worldmodel.wm.utils import load_pretrained
import json
from pathlib import Path
import torch

device = 'mps'

#model = load_pretrained('quentinll/lewm-pusht').to(device).eval()
model = swm.wm.utils.load_pretrained('models--quentinll--lewm-pusht/weights.pt').to(device).eval()
model.requires_grad_(False)
model.interpolate_pos_encoding = True   # official eval sets this

# inspect what the official checkpoint declares:
import json
from pathlib import Path
cdir = Path(swm.data.utils.get_cache_dir(sub_folder='checkpoints'))
# the HF download lands under a user/repo path; find its config.json
for cfgp in cdir.rglob('config.json'):
    if 'lewm-pusht' in str(cfgp) or 'quentinll' in str(cfgp):
        oc = json.loads(cfgp.read_text())
        enc = oc.get('encoder', oc.get('model', {}).get('encoder', {}))
        print('img/patch:', enc.get('image_size'), enc.get('patch_size'))
        break

import stable_pretraining as spt
def img_transform(img_size, dtype=torch.float32):
    return transforms.Compose([
        transforms.ToImage(),
        transforms.ToDtype(dtype, scale=True),                    # /255 → [0,1]
        transforms.Normalize(**spt.data.dataset_stats.ImageNet),  # ImageNet mean/std
        transforms.Resize(size=img_size),
    ])
transform = {'pixels': img_transform(224), 'goal': img_transform(224)}

# 2) world image_shape matches
world = swm.World('swm/PushT-v1', num_envs=8, image_shape=(224, 224), max_episode_steps=100)

solver = CEMSolver(
    model=model,
    num_samples=300,
    n_steps=5,
    device=device,
)

from sklearn import preprocessing
import numpy as np
ds = swm.data.load_dataset('pusht_expert_train.lance')  # or .h5 — the official PushT dataset, not your 64-ep tutorial set
process = {}
for col in ['action', 'proprio', 'state']:
    sc = preprocessing.StandardScaler()
    d = ds.get_col_data(col); d = d[~np.isnan(d).any(axis=1)]
    sc.fit(d); process[col] = sc
    if col != 'action': process[f'goal_{col}'] = process[col]

policy = WorldModelPolicy(
    solver=solver,
    config=PlanConfig(
        horizon=5,
        receding_horizon=5,
        action_block=5,
        warm_start=True,
    ),
    transform=transform,
    process=process
)

world.set_policy(policy)
results = world.evaluate(episodes=32, seed=0, video='videos/tutorial_eval')
world.close()

print(results)


19:38:02 | INFO  | utils.py    | Created ViT-tiny from scratch with config: {'hidden_size': 192, 'num_hidden_layers': 12, 'num_attention_heads': 3, 'intermediate_size': 768, 'image_size': 224, 'patch_size': 14}


RuntimeError: Error(s) in loading state_dict for LeWM:
	Missing key(s) in state_dict: "encoder.layers.0.attention.q_proj.weight", "encoder.layers.0.attention.q_proj.bias", "encoder.layers.0.attention.k_proj.weight", "encoder.layers.0.attention.k_proj.bias", "encoder.layers.0.attention.v_proj.weight", "encoder.layers.0.attention.v_proj.bias", "encoder.layers.0.attention.o_proj.weight", "encoder.layers.0.attention.o_proj.bias", "encoder.layers.0.layernorm_before.weight", "encoder.layers.0.layernorm_before.bias", "encoder.layers.0.layernorm_after.weight", "encoder.layers.0.layernorm_after.bias", "encoder.layers.0.mlp.fc1.weight", "encoder.layers.0.mlp.fc1.bias", "encoder.layers.0.mlp.fc2.weight", "encoder.layers.0.mlp.fc2.bias", "encoder.layers.1.attention.q_proj.weight", "encoder.layers.1.attention.q_proj.bias", "encoder.layers.1.attention.k_proj.weight", "encoder.layers.1.attention.k_proj.bias", "encoder.layers.1.attention.v_proj.weight", "encoder.layers.1.attention.v_proj.bias", "encod